In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# Define the folder path and the column names
# '/Users/jul/Desktop/uni/Data Analytics/project/PROBE-202411'
# /Users/jul/Desktop/uni/Data Analytics/PROBE-202409
# '/Users/jul/Desktop/uni/Data Analytics/PROBE-202412'
folder_paths = [
    '/Users/jul/Desktop/uni/Data Analytics/PROBE-202412'
]
columns = [
    'VehicleID',
    'gpsvalid',
    'lat',
    'lon',
    'timestamp',
    'speed',
    'heading',
    'for_hire_light',
    'engine_acc'
]

In [3]:
# Initialize an empty list to store the dataframes
all_dfs = []

In [4]:

# Loop through all folders
for folder_path in folder_paths:
    # Loop through all files in the directory
    for filename in os.listdir(folder_path):
        # Check if the file is a CSV file
        if filename.endswith('.csv.out'):
            file_path = os.path.join(folder_path, filename)

            # Read the CSV file into a dataframe with the specified column names
            df = pd.read_csv(file_path, names=columns)

            # Append the dataframe to the list
            all_dfs.append(df)

# Concatenate all dataframes in the list into a single dataframe
combined_df = pd.concat(all_dfs, ignore_index=True)

In [5]:
combined_df

,VehicleID,gpsvalid,lat,lon,timestamp,speed,heading,for_hire_light,engine_acc
0,V8F0cK98HyTMrcSdeR47rexj0rw,1,13.73413,100.68927,2024-12-21 23:55:57,107,96,0,1
1,r8yh9EoluJ0u53F81KhXcZ8V6VQ,1,13.68293,100.46149,2024-12-21 23:56:41,0,348,0,0
2,fhAmmgsL3IogEfBatvNCwSLr04A,1,13.65472,100.30120,2024-12-21 23:57:46,0,1,0,0
3,Cggse+7450jI5U3HLEOfhoBCOGk,1,13.86938,100.46595,2024-12-21 23:57:50,0,110,0,0
4,gLq3BOZ8ggjCHmjhqWAQ7ZyVBlg,1,13.77983,100.39398,2024-12-21 23:58:03,0,302,0,0
...,...,...,...,...,...,...,...,...,...
59432987,YKvdFXsrD9SJiakUQEpwxEOaK1Q,1,13.86524,100.58894,2024-12-27 23:59:10,0,273,0,0
59432988,wsK20gYyyNpTfG3s/1IuVbg32WA,1,7.89154,98.38982,2024-12-27 23:59:10,35,352,0,1
59432989,s+FMc1WPBo2oVF06gx27GDr29G8,1,7.84152,98.30315,2024-12-27 23:59:10,0,177,0,0
59432990,FWfSz92y9h30nkpZ/xUTTwLPgkE,1,13.78749,100.64972,2024-12-27 23:59:10,59,33,0,1


In [6]:
cleaned_df = combined_df.dropna()


In [7]:
cleaned_taxi_df = cleaned_df[
    (combined_df['gpsvalid'] == 1) &
    (combined_df['engine_acc'] == 1) &
    (combined_df['lat'].between(-90, 90)) &
    (combined_df['lon'].between(-180, 180)) &
    (combined_df['speed'] >= 0)
].copy().reset_index(drop=True)

In [8]:
# Find the unique VehicleIDs of every vehicle that EVER reported for_hire_light = 1
# This is our most reliable definition of a taxi.
taxi_ids = cleaned_taxi_df[cleaned_taxi_df['for_hire_light'] == 1]['VehicleID'].unique()

In [9]:
# Now, filter the main dataframe to keep ALL records for these identified taxis
# This gives us their full journey, not just when their light was on.
taxis_df = cleaned_taxi_df[cleaned_taxi_df['VehicleID'].isin(taxi_ids)].copy()

In [10]:
# The 'timestamp' column is just text right now. We need to convert it.
taxis_df['timestamp'] = pd.to_datetime(taxis_df['timestamp'])

Sort Data

In [11]:
# Sort by the vehicle first, then by the time.
taxis_df.sort_values(by=['VehicleID', 'timestamp'], inplace=True)

In [12]:
# Reset the index after sorting for a clean DataFrame
taxis_df.reset_index(drop=True, inplace=True)

In [13]:
# Extract time-based features from the timestamp
taxis_df['hour'] = taxis_df['timestamp'].dt.hour
taxis_df['day_of_week'] = taxis_df['timestamp'].dt.dayofweek # Monday=0, Sunday=6
taxis_df['is_weekend'] = (taxis_df['day_of_week'] >= 5).astype(int)

In [14]:
# --- Advanced: Identify Trips ---
# A trip starts when the 'for_hire_light' changes from 1 (empty) to 0 (occupied).
# We can create a 'trip_id' for each taxi.

# First, detect the change from 1 to 0
taxis_df['trip_start'] = (taxis_df['for_hire_light'].shift(1) == 1) & (taxis_df['for_hire_light'] == 0)

In [15]:
# We also need to ensure it's the same vehicle
taxis_df['trip_start'] = taxis_df['trip_start'] & (taxis_df['VehicleID'].shift(1) == taxis_df['VehicleID'])

In [16]:
# Now create a unique ID for each trip using a cumulative sum
taxis_df['trip_id'] = taxis_df.groupby('VehicleID')['trip_start'].cumsum()

In [17]:
# Let's clean up the intermediate column
taxis_df.drop(columns=['trip_start'], inplace=True)

In [18]:
taxis_df

,VehicleID,gpsvalid,lat,lon,timestamp,speed,heading,for_hire_light,engine_acc,hour,day_of_week,is_weekend,trip_id
0,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64478,100.63768,2024-12-02 08:34:52,0,100,1,1,8,0,0,0
1,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64478,100.63768,2024-12-02 08:36:52,0,100,1,1,8,0,0,0
2,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64490,100.63728,2024-12-02 08:38:52,11,219,1,1,8,0,0,0
3,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64066,100.63463,2024-12-02 08:39:52,0,210,1,1,8,0,0,0
4,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64285,100.63578,2024-12-02 08:42:52,12,25,1,1,8,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
17048599,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.93001,100.49104,2024-12-31 21:51:50,48,44,1,1,21,1,0,0
17048600,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.93532,100.49610,2024-12-31 21:52:50,45,32,1,1,21,1,0,0
17048601,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94625,100.49688,2024-12-31 21:56:52,6,206,1,1,21,1,0,0
17048602,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94636,100.49665,2024-12-31 21:58:51,0,206,1,1,21,1,0,0


In [19]:
# cars with trip id
taxis_df.groupby('VehicleID')["trip_id"].nunique()

VehicleID
++qQzutWwL31NcUo8s0jiGZzzS0      1
+1indEOKr/ikPVrJQTVjw4FGxBE      1
+20prWr63K5svsMtTmdqLnmsTGE    287
+A3arvBS15eOvdHE+E06+Ng28+E    145
+BAgYWCbvz0z377ef3Yp687Pp+0    458
                              ... 
zt6r6X6cjBVDqWxKcbyXkn1Cydw    472
zvkvMBxj2VDpNjIfmifwwxl6nMo     79
zwNAI8pHuCgPOF5NMs2nVaXBG04    218
zyy0Hv5yMoClNPQlx1tR4NBssFI      1
zzLYPcDONaA8lLF2aJYFKnoRDQ4      1
Name: trip_id, Length: 2099, dtype: int64

In [20]:
#how many taxis extracted
taxis_df['VehicleID'].nunique()

2099

In [21]:
#how many trips extracted
taxis_df.groupby('VehicleID')['trip_id'].max().sum()

np.int64(458878)

Calculate Idle Time

In [22]:
# It creates the unique ID for each continuous idle period.
taxis_df['idle_start'] = (taxis_df['for_hire_light'].shift(1) == 0) & \
                         (taxis_df['for_hire_light'] == 1) & \
                         (taxis_df['VehicleID'].shift(1) == taxis_df['VehicleID'])
taxis_df['idle_period_id'] = taxis_df.groupby('VehicleID')['idle_start'].cumsum()
taxis_df.loc[taxis_df['trip_id'] == 0, 'idle_period_id'] = 0

In [23]:
# Create a temporary DataFrame with only the idle data points
idle_periods_df = taxis_df[taxis_df['for_hire_light'] == 1].copy()

# Group by each unique idle period to calculate its duration
idle_summary = idle_periods_df.groupby(['VehicleID', 'idle_period_id']).agg(
    start_time=('timestamp', 'min'),
    end_time=('timestamp', 'max')
).reset_index()

# Calculate the duration in minutes
idle_summary['idle_duration_minutes'] = (idle_summary['end_time'] - idle_summary['start_time']).dt.total_seconds() / 60 + 1

print("--- This is the information we will merge back ---")
display(idle_summary[['VehicleID', 'idle_period_id', 'idle_duration_minutes']].head())

--- This is the information we will merge back ---


,VehicleID,idle_period_id,idle_duration_minutes
0,++qQzutWwL31NcUo8s0jiGZzzS0,0,37340.483333
1,+1indEOKr/ikPVrJQTVjw4FGxBE,0,44041.950000
2,+20prWr63K5svsMtTmdqLnmsTGE,0,34.000000
3,+20prWr63K5svsMtTmdqLnmsTGE,2,28.000000
4,+20prWr63K5svsMtTmdqLnmsTGE,3,11.783333


In [24]:
# We use a 'left' merge to ensure we keep ALL original rows from taxis_df
# The merge will add the 'idle_duration_minutes' from the summary to the main table
# based on the matching VehicleID and idle_period_id.
taxis_df = pd.merge(
    taxis_df,
    idle_summary[['VehicleID', 'idle_period_id', 'idle_duration_minutes']],
    on=['VehicleID', 'idle_period_id'],
    how='left'
)

# After merging, the rows that were NOT idle (i.e., part of a trip) will have NaN
# for the idle duration. We should fill these with 0.
taxis_df['idle_duration_minutes'].fillna(0, inplace=True)

/var/folders/qb/yf4211gs4kv7hy5d5jp7q6bw0000gn/T/ipykernel_2481/1755066191.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  taxis_df['idle_duration_minutes'].fillna(0, inplace=True)


In [25]:
taxis_df.drop(columns=['idle_start'], inplace=True)

In [26]:
taxis_df

,VehicleID,gpsvalid,lat,lon,timestamp,speed,heading,for_hire_light,engine_acc,hour,day_of_week,is_weekend,trip_id,idle_period_id,idle_duration_minutes
0,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64478,100.63768,2024-12-02 08:34:52,0,100,1,1,8,0,0,0,0,37340.483333
1,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64478,100.63768,2024-12-02 08:36:52,0,100,1,1,8,0,0,0,0,37340.483333
2,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64490,100.63728,2024-12-02 08:38:52,11,219,1,1,8,0,0,0,0,37340.483333
3,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64066,100.63463,2024-12-02 08:39:52,0,210,1,1,8,0,0,0,0,37340.483333
4,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64285,100.63578,2024-12-02 08:42:52,12,25,1,1,8,0,0,0,0,37340.483333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17048599,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.93001,100.49104,2024-12-31 21:51:50,48,44,1,1,21,1,0,0,0,44099.000000
17048600,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.93532,100.49610,2024-12-31 21:52:50,45,32,1,1,21,1,0,0,0,44099.000000
17048601,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94625,100.49688,2024-12-31 21:56:52,6,206,1,1,21,1,0,0,0,44099.000000
17048602,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94636,100.49665,2024-12-31 21:58:51,0,206,1,1,21,1,0,0,0,44099.000000


Filter out irrelavant date and time

In [27]:
unique_years = taxis_df['timestamp'].dt.year.unique()
unique_years.sort()
unique_years

array([1970, 2024], dtype=int32)

In [28]:
# filter out years more than 2024 and less than 2015
year_series = taxis_df['timestamp'].dt.year

# Keep only the rows where the year is between 2016 and 2024 (inclusive)
# "later than 2015" means >= 2016
# "drop further than 2024" means <= 2024

taxis_df = taxis_df[(year_series == 2024)].copy()

In [29]:
taxis_df

,VehicleID,gpsvalid,lat,lon,timestamp,speed,heading,for_hire_light,engine_acc,hour,day_of_week,is_weekend,trip_id,idle_period_id,idle_duration_minutes
0,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64478,100.63768,2024-12-02 08:34:52,0,100,1,1,8,0,0,0,0,37340.483333
1,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64478,100.63768,2024-12-02 08:36:52,0,100,1,1,8,0,0,0,0,37340.483333
2,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64490,100.63728,2024-12-02 08:38:52,11,219,1,1,8,0,0,0,0,37340.483333
3,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64066,100.63463,2024-12-02 08:39:52,0,210,1,1,8,0,0,0,0,37340.483333
4,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64285,100.63578,2024-12-02 08:42:52,12,25,1,1,8,0,0,0,0,37340.483333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17048599,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.93001,100.49104,2024-12-31 21:51:50,48,44,1,1,21,1,0,0,0,44099.000000
17048600,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.93532,100.49610,2024-12-31 21:52:50,45,32,1,1,21,1,0,0,0,44099.000000
17048601,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94625,100.49688,2024-12-31 21:56:52,6,206,1,1,21,1,0,0,0,44099.000000
17048602,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94636,100.49665,2024-12-31 21:58:51,0,206,1,1,21,1,0,0,0,44099.000000


In [30]:
taxis_df.groupby('trip_id')["VehicleID"].nunique()

trip_id
0      2099
1      1570
2      1533
3      1513
4      1494
       ... 
913       1
914       1
915       1
916       1
917       1
Name: VehicleID, Length: 918, dtype: int64

Filter to Just BMR Bangkok

In [31]:
#Define Regions
BKK_REGION_BOUNDS = {
    'min_lat': 13.4,
    'max_lat': 14.2,
    'min_lon': 99.9,
    'max_lon': 101.3
}

In [32]:
taxis_df_bkk = taxis_df[
    (taxis_df['lat'] >= BKK_REGION_BOUNDS['min_lat']) &
    (taxis_df['lat'] <= BKK_REGION_BOUNDS['max_lat']) &
    (taxis_df['lon'] >= BKK_REGION_BOUNDS['min_lon']) &
    (taxis_df['lon'] <= BKK_REGION_BOUNDS['max_lon'])
].copy()

In [33]:
taxis_df_bkk

,VehicleID,gpsvalid,lat,lon,timestamp,speed,heading,for_hire_light,engine_acc,hour,day_of_week,is_weekend,trip_id,idle_period_id,idle_duration_minutes
0,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64478,100.63768,2024-12-02 08:34:52,0,100,1,1,8,0,0,0,0,37340.483333
1,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64478,100.63768,2024-12-02 08:36:52,0,100,1,1,8,0,0,0,0,37340.483333
2,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64490,100.63728,2024-12-02 08:38:52,11,219,1,1,8,0,0,0,0,37340.483333
3,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64066,100.63463,2024-12-02 08:39:52,0,210,1,1,8,0,0,0,0,37340.483333
4,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64285,100.63578,2024-12-02 08:42:52,12,25,1,1,8,0,0,0,0,37340.483333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17048599,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.93001,100.49104,2024-12-31 21:51:50,48,44,1,1,21,1,0,0,0,44099.000000
17048600,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.93532,100.49610,2024-12-31 21:52:50,45,32,1,1,21,1,0,0,0,44099.000000
17048601,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94625,100.49688,2024-12-31 21:56:52,6,206,1,1,21,1,0,0,0,44099.000000
17048602,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94636,100.49665,2024-12-31 21:58:51,0,206,1,1,21,1,0,0,0,44099.000000


In [34]:
#filter speed
taxis_df_bkk = taxis_df_bkk[taxis_df_bkk['speed'] <= 180].copy()

In [35]:
taxis_df_bkk

,VehicleID,gpsvalid,lat,lon,timestamp,speed,heading,for_hire_light,engine_acc,hour,day_of_week,is_weekend,trip_id,idle_period_id,idle_duration_minutes
0,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64478,100.63768,2024-12-02 08:34:52,0,100,1,1,8,0,0,0,0,37340.483333
1,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64478,100.63768,2024-12-02 08:36:52,0,100,1,1,8,0,0,0,0,37340.483333
2,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64490,100.63728,2024-12-02 08:38:52,11,219,1,1,8,0,0,0,0,37340.483333
3,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64066,100.63463,2024-12-02 08:39:52,0,210,1,1,8,0,0,0,0,37340.483333
4,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64285,100.63578,2024-12-02 08:42:52,12,25,1,1,8,0,0,0,0,37340.483333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17048599,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.93001,100.49104,2024-12-31 21:51:50,48,44,1,1,21,1,0,0,0,44099.000000
17048600,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.93532,100.49610,2024-12-31 21:52:50,45,32,1,1,21,1,0,0,0,44099.000000
17048601,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94625,100.49688,2024-12-31 21:56:52,6,206,1,1,21,1,0,0,0,44099.000000
17048602,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94636,100.49665,2024-12-31 21:58:51,0,206,1,1,21,1,0,0,0,44099.000000


In [36]:
#how many taxis extracted
taxis_df_bkk['VehicleID'].nunique()

2021

In [37]:
taxis_df_bkk.groupby('VehicleID')['trip_id'].max().sum()

np.int64(457471)

In [38]:
taxis_df_bkk.columns

Index(['VehicleID', 'gpsvalid', 'lat', 'lon', 'timestamp', 'speed', 'heading',
       'for_hire_light', 'engine_acc', 'hour', 'day_of_week', 'is_weekend',
       'trip_id', 'idle_period_id', 'idle_duration_minutes'],
      dtype='object')

In [39]:
# Save as CSV
taxis_df_bkk.to_csv("cleaned_taxis_11_2024.csv", index=False)

In [40]:
# Example: read data and compute fare
df = pd.read_csv('cleaned_taxis_11_2024.csv')

# Ensure timestamp is in datetime format and sort the data
df['timestamp'] = pd.to_datetime(df['timestamp'])
# We must sort by VehicleID first, then trip_id, then timestamp
df = df.sort_values(by=['VehicleID', 'trip_id', 'timestamp'])


In [41]:
def haversine_distance(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance in kilometers between two points
    on the earth (specified in decimal degrees).
    """
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371 # Radius of earth in kilometers.
    return c * r

In [42]:
# --- Part 1: Calculate distance between each point (same as before) ---
df['lat_next'] = df.groupby(['VehicleID', 'trip_id'])['lat'].shift(-1)
df['lon_next'] = df.groupby(['VehicleID', 'trip_id'])['lon'].shift(-1)

# This column is the distance between one point and the next, not the total
df['segment_distance_km'] = haversine_distance(df['lon'], df['lat'], df['lon_next'], df['lat_next'])

# --- Part 2: Calculate the TOTAL distance for the whole trip (same as before) ---
# This creates a summary table with one row per trip and its total distance
trip_distances = df.groupby(['VehicleID', 'trip_id'])['segment_distance_km'].sum().reset_index()
trip_distances.rename(columns={'segment_distance_km': 'total_trip_distance_km'}, inplace=True)


# --- Part 3: Merge the total trip distance back to the main dataframe ---
# This is the new step you were looking for.
df = pd.merge(df, trip_distances, on=['VehicleID', 'trip_id'], how='left')

#drop all rows that has trip distance that more than 300 km
df = df[df['total_trip_distance_km'] <= 300]

#drop all rows that has trip distance that equal to 0 km
df = df[df['total_trip_distance_km'] > 0]

In [43]:
def calculate_total_trip_fees(distance_km):
    """
    Calculates the taxi fare in THB based on the provided distance in kilometers,
    using the official Thai taxi rate structure.
    """
    # Handle cases with no or very small distance
    if distance_km <= 0:
        return 0

    # First 1 km
    fare = 35.0

    if distance_km > 1:
        # km >1 to 10
        d = min(distance_km, 10) - 1
        fare += d * 6.50

    if distance_km > 10:
        # km >10 to 20
        d = min(distance_km, 20) - 10
        fare += d * 7.00

    if distance_km > 20:
        # km >20 to 40
        d = min(distance_km, 40) - 20
        fare += d * 8.00

    if distance_km > 40:
        # km >40 to 60
        d = min(distance_km, 60) - 40
        fare += d * 8.50

    if distance_km > 60:
        # km >60 to 80
        d = min(distance_km, 80) - 60
        fare += d * 9.00

    if distance_km > 80:
        # km >80
        d = distance_km - 80
        fare += d * 10.50

    return fare

In [44]:
# (Assuming you have already defined the calculate_total_trip_fees function)
df['total_trip_fees'] = df['total_trip_distance_km'].apply(calculate_total_trip_fees)

In [45]:
# getting all important features
# --- We will group by each unique trip to calculate the new features ---
trip_groups = df.groupby(['VehicleID', 'trip_id'])

# --- 1. Calculate Trip Duration ---
# Get the start and end time for each trip
trip_start_time = trip_groups['timestamp'].min()
trip_end_time = trip_groups['timestamp'].max()

# Calculate duration in seconds and then minutes
duration_seconds = (trip_end_time - trip_start_time).dt.total_seconds()
duration_minutes = duration_seconds / 60

# --- 2. Extract Pickup Time Features ---
# We use the start time for this
pickup_hour = trip_start_time.dt.hour
pickup_dayofweek = trip_start_time.dt.dayofweek # Monday=0, Sunday=6

# --- 3. Calculate Average Speed ---
average_speed = trip_groups['speed'].mean()

# --- 4. Get Start and End Locations ---
start_lat = trip_groups['lat'].first()
start_lon = trip_groups['lon'].first()
end_lat = trip_groups['lat'].last()
end_lon = trip_groups['lon'].last()

# --- 5. Assemble the new features into a single DataFrame ---
features_df = pd.DataFrame({
    'duration_minutes': duration_minutes,
    'pickup_hour': pickup_hour,
    'pickup_dayofweek': pickup_dayofweek,
    'average_speed': average_speed,
    'start_lat': start_lat,
    'start_lon': start_lon,
    'end_lat': end_lat,
    'end_lon': end_lon
})

In [46]:
# Now, merge the features into your main 'df'
# The 'on' parameter tells pandas to match rows using VehicleID and trip_id
# The 'how='left'' ensures you keep all rows from your original df
features_df_to_merge = features_df.reset_index()
df = pd.merge(df, features_df_to_merge, on=['VehicleID', 'trip_id'], how='left')

In [47]:
df
df = df.dropna()

In [48]:
df

,VehicleID,gpsvalid,lat,lon,timestamp,speed,heading,for_hire_light,engine_acc,hour,...,total_trip_distance_km,total_trip_fees,duration_minutes,pickup_hour,pickup_dayofweek,average_speed,start_lat,start_lon,end_lat,end_lon
0,+20prWr63K5svsMtTmdqLnmsTGE,1,13.66729,100.58770,2024-12-01 05:24:13,0,290,0,1,5,...,36.505692,295.545534,60.866667,5,6,38.090909,13.66729,100.58770,13.69314,100.75144
1,+20prWr63K5svsMtTmdqLnmsTGE,1,13.66738,100.58770,2024-12-01 05:26:05,0,290,0,1,5,...,36.505692,295.545534,60.866667,5,6,38.090909,13.66729,100.58770,13.69314,100.75144
2,+20prWr63K5svsMtTmdqLnmsTGE,1,13.66738,100.58770,2024-12-01 05:28:05,0,290,0,1,5,...,36.505692,295.545534,60.866667,5,6,38.090909,13.66729,100.58770,13.69314,100.75144
3,+20prWr63K5svsMtTmdqLnmsTGE,1,13.66734,100.58778,2024-12-01 05:30:05,0,290,0,1,5,...,36.505692,295.545534,60.866667,5,6,38.090909,13.66729,100.58770,13.69314,100.75144
4,+20prWr63K5svsMtTmdqLnmsTGE,1,13.66734,100.58778,2024-12-01 05:31:05,0,290,0,1,5,...,36.505692,295.545534,60.866667,5,6,38.090909,13.66729,100.58770,13.69314,100.75144
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12108883,zwNAI8pHuCgPOF5NMs2nVaXBG04,1,13.73320,100.38991,2024-12-26 17:30:57,24,81,1,1,17,...,48.191423,393.127093,763.600000,4,3,43.368421,13.91573,100.59949,13.77508,100.39478
12108884,zwNAI8pHuCgPOF5NMs2nVaXBG04,1,13.73133,100.39144,2024-12-26 17:31:58,31,91,1,1,17,...,48.191423,393.127093,763.600000,4,3,43.368421,13.91573,100.59949,13.77508,100.39478
12108885,zwNAI8pHuCgPOF5NMs2nVaXBG04,1,13.73560,100.39541,2024-12-26 17:33:57,58,359,1,1,17,...,48.191423,393.127093,763.600000,4,3,43.368421,13.91573,100.59949,13.77508,100.39478
12108886,zwNAI8pHuCgPOF5NMs2nVaXBG04,1,13.75033,100.39522,2024-12-26 17:35:58,52,359,1,1,17,...,48.191423,393.127093,763.600000,4,3,43.368421,13.91573,100.59949,13.77508,100.39478


In [49]:
df.columns

Index(['VehicleID', 'gpsvalid', 'lat', 'lon', 'timestamp', 'speed', 'heading',
       'for_hire_light', 'engine_acc', 'hour', 'day_of_week', 'is_weekend',
       'trip_id', 'idle_period_id', 'idle_duration_minutes', 'lat_next',
       'lon_next', 'segment_distance_km', 'total_trip_distance_km',
       'total_trip_fees', 'duration_minutes', 'pickup_hour',
       'pickup_dayofweek', 'average_speed', 'start_lat', 'start_lon',
       'end_lat', 'end_lon'],
      dtype='object')

Feature Engineering


In [50]:
from sklearn.cluster import MiniBatchKMeans
import pandas as pd

# Assuming 'df' is your main dataframe
# Create 100 location clusters. You can experiment with this number.
kmeans = MiniBatchKMeans(n_clusters=100, batch_size=10000, random_state=42, n_init=10)
df['zone_id'] = kmeans.fit_predict(df[['lat', 'lon']])

# First, identify pickup and dropoff events
df['pickup_flag'] = ((df['for_hire_light'].shift(1) == 1) & (df['for_hire_light'] == 0)).astype(int)
df['dropoff_flag'] = ((df['for_hire_light'].shift(1) == 0) & (df['for_hire_light'] == 1)).astype(int)

# 1. Check if 'timestamp' is a column before trying to set it as the index
if 'timestamp' in df.columns:
    df.set_index('timestamp', inplace=True)

# Sort the DataFrame by the timestamp index before doing the rolling calculation
df.sort_index(inplace=True)

# By adding .values, we assign the results directly, ignoring the index alignment issue.
# This is safe because the dataframe is already sorted.
df['pickups_in_zone_last_hr'] = df.groupby('zone_id')['pickup_flag'].rolling('1H').sum().reset_index(0, drop=True).values
df['dropoffs_in_zone_last_hr'] = df.groupby('zone_id')['dropoff_flag'].rolling('1H').sum().reset_index(0, drop=True).values

# Fill any NaNs that result from the rolling window
df.fillna({'pickups_in_zone_last_hr': 0, 'dropoffs_in_zone_last_hr': 0}, inplace=True)

# Reset index to bring 'timestamp' back as a column
df.reset_index(inplace=True)

# Create a feature for 'zone_id' combined with the hour
df['zone_hour_interaction'] = df['zone_id'].astype(str) + '_' + df['hour'].astype(str)

# Now, label encode this new categorical feature so the model can use it
df['zone_hour_interaction'] = df['zone_hour_interaction'].astype('category').cat.codes

# Ensure data is sorted for accurate lag features
df.sort_values(by=['VehicleID', 'timestamp'], inplace=True)

# --- THE FIX IS HERE ---
# Apply .rolling() to the GroupBy object BEFORE selecting the 'speed' column.
# Then, select 'speed' and calculate the mean.
rolling_avg_speed = df.groupby('VehicleID').rolling('15T', on='timestamp')['speed'].mean()

# The result of a rolling operation on a groupby is a Series with a MultiIndex.
# We need to drop the 'VehicleID' level of the index to align it back to the original df.
df['avg_speed_last_15m'] = rolling_avg_speed.reset_index(level=0, drop=True).values
# --- END OF FIX ---


# Fill NaNs for the start of each taxi's journey
# Using 'bfill' (backfill) is a good choice here to fill the initial nulls.
df['avg_speed_last_15m'].fillna(method='bfill', inplace=True)

/var/folders/qb/yf4211gs4kv7hy5d5jp7q6bw0000gn/T/ipykernel_2481/608406458.py:22: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df['pickups_in_zone_last_hr'] = df.groupby('zone_id')['pickup_flag'].rolling('1H').sum().reset_index(0, drop=True).values
/var/folders/qb/yf4211gs4kv7hy5d5jp7q6bw0000gn/T/ipykernel_2481/608406458.py:23: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df['dropoffs_in_zone_last_hr'] = df.groupby('zone_id')['dropoff_flag'].rolling('1H').sum().reset_index(0, drop=True).values
/var/folders/qb/yf4211gs4kv7hy5d5jp7q6bw0000gn/T/ipykernel_2481/608406458.py:43: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  rolling_avg_speed = df.groupby('VehicleID').rolling('15T', on='timestamp')['speed'].mean()
/var/folders/qb/yf4211gs4kv7hy5d5jp7q6bw0000gn/T/ipykernel_2481/608406458.py:43: FutureWarning: 'T' is deprecated and 

In [51]:
df

,timestamp,VehicleID,gpsvalid,lat,lon,speed,heading,for_hire_light,engine_acc,hour,...,start_lon,end_lat,end_lon,zone_id,pickup_flag,dropoff_flag,pickups_in_zone_last_hr,dropoffs_in_zone_last_hr,zone_hour_interaction,avg_speed_last_15m
44126,2024-12-01 05:24:13,+20prWr63K5svsMtTmdqLnmsTGE,1,13.66729,100.58770,0,290,0,1,5,...,100.58770,13.69314,100.75144,21,0,0,21.0,20.0,331,0.000000
44423,2024-12-01 05:26:05,+20prWr63K5svsMtTmdqLnmsTGE,1,13.66738,100.58770,0,290,0,1,5,...,100.58770,13.69314,100.75144,21,0,0,13.0,16.0,331,0.000000
44719,2024-12-01 05:28:05,+20prWr63K5svsMtTmdqLnmsTGE,1,13.66738,100.58770,0,290,0,1,5,...,100.58770,13.69314,100.75144,21,0,0,12.0,19.0,331,0.000000
45039,2024-12-01 05:30:05,+20prWr63K5svsMtTmdqLnmsTGE,1,13.66734,100.58778,0,290,0,1,5,...,100.58770,13.69314,100.75144,21,0,0,7.0,13.0,331,0.000000
45134,2024-12-01 05:31:05,+20prWr63K5svsMtTmdqLnmsTGE,1,13.66734,100.58778,0,290,0,1,5,...,100.58770,13.69314,100.75144,21,0,0,4.0,12.0,331,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9640311,2024-12-26 17:30:57,zwNAI8pHuCgPOF5NMs2nVaXBG04,1,13.73320,100.38991,24,81,1,1,17,...,100.59949,13.77508,100.39478,51,0,0,14.0,12.0,1113,31.333333
9640584,2024-12-26 17:31:58,zwNAI8pHuCgPOF5NMs2nVaXBG04,1,13.73133,100.39144,31,91,1,1,17,...,100.59949,13.77508,100.39478,51,0,0,16.0,16.0,1113,31.250000
9641386,2024-12-26 17:33:57,zwNAI8pHuCgPOF5NMs2nVaXBG04,1,13.73560,100.39541,58,359,1,1,17,...,100.59949,13.77508,100.39478,51,0,0,9.0,16.0,1113,36.600000
9642101,2024-12-26 17:35:58,zwNAI8pHuCgPOF5NMs2nVaXBG04,1,13.75033,100.39522,52,359,1,1,17,...,100.59949,13.77508,100.39478,60,0,0,7.0,7.0,1353,39.166667


In [52]:
df.columns

Index(['timestamp', 'VehicleID', 'gpsvalid', 'lat', 'lon', 'speed', 'heading',
       'for_hire_light', 'engine_acc', 'hour', 'day_of_week', 'is_weekend',
       'trip_id', 'idle_period_id', 'idle_duration_minutes', 'lat_next',
       'lon_next', 'segment_distance_km', 'total_trip_distance_km',
       'total_trip_fees', 'duration_minutes', 'pickup_hour',
       'pickup_dayofweek', 'average_speed', 'start_lat', 'start_lon',
       'end_lat', 'end_lon', 'zone_id', 'pickup_flag', 'dropoff_flag',
       'pickups_in_zone_last_hr', 'dropoffs_in_zone_last_hr',
       'zone_hour_interaction', 'avg_speed_last_15m'],
      dtype='object')

In [53]:
# Paste this whole cell into your notebook (replaces previous functions).
import math
from datetime import timedelta
from typing import Iterable, Optional, Tuple, List

import numpy as np
import pandas as pd

# Optional: avoid installing packages during heavy processing — prefer pre-install
try:
    import h3
except Exception as e:
    raise ImportError("Missing 'h3' package. Install it in your environment (pip install h3) before running.") from e

# H3 wrappers (compatible with h3 v3/v4)
def _h3_cell(lat: float, lon: float, res: int) -> Optional[str]:
    if pd.isna(lat) or pd.isna(lon):
        return None
    if hasattr(h3, "geo_to_h3"):
        return h3.geo_to_h3(lat, lon, res)
    if hasattr(h3, "latlng_to_cell"):
        return h3.latlng_to_cell(lat, lon, res)
    raise AttributeError("No suitable H3 function for lat/lon -> cell")

def _h3_neighbors(cell: str, k: int) -> List[str]:
    if cell is None:
        return []
    if hasattr(h3, "k_ring"):
        return list(h3.k_ring(cell, k))
    if hasattr(h3, "grid_disk"):
        return list(h3.grid_disk(cell, k))
    return [cell]

# --------------------------
# Helpers
# --------------------------
def _as_5min_floor(ts: pd.Series) -> pd.Series:
    return (ts.dt.floor("5min")).astype("datetime64[ns]")

def _sin_cos(series: pd.Series, period: float) -> Tuple[pd.Series, pd.Series]:
    radians = 2 * math.pi * (series.astype(float) / period)
    return np.sin(radians), np.cos(radians)

def build_complete_index(time_bins: Iterable[pd.Timestamp], cells: Iterable[str]) -> pd.MultiIndex:
    return pd.MultiIndex.from_product([pd.Index(time_bins, name="time_bin"), pd.Index(cells, name="h3_cell")])

# --------------------------
# Core functions (safe)
# --------------------------
def assign_h3_cells(
    df: pd.DataFrame,
    lat_col: str,
    lon_col: str,
    h3_res: int = 7,
    out_col: str = "h3_cell",
) -> pd.DataFrame:
    df = df.copy()
    # vectorized-ish mapping using list comprehension (safe)
    coords = df[[lat_col, lon_col]].to_numpy()
    df[out_col] = [_h3_cell(lat, lon, h3_res) for lat, lon in coords]
    return df

def aggregate_events_to_cells(
    df: pd.DataFrame,
    ts_col: str,
    cell_col: str,
    bin_minutes: int = 5,
    value_col: Optional[str] = None,
    agg_name: str = "count",
    complete_index: Optional[pd.MultiIndex] = None,
) -> pd.DataFrame:
    df = df.copy()
    df["time_bin"] = _as_5min_floor(pd.to_datetime(df[ts_col]))
    if value_col is None or agg_name == "count":
        grouped = df.groupby(["time_bin", cell_col]).size().rename("value").to_frame()
    else:
        grouped = df.groupby(["time_bin", cell_col])[value_col].agg(agg_name).rename("value").to_frame()
    if complete_index is not None:
        grouped = grouped.reindex(complete_index, fill_value=0)
    return grouped.reset_index()

def add_time_features(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame["hour"] = frame["time_bin"].dt.hour
    frame["dow"] = frame["time_bin"].dt.weekday
    frame["is_weekend"] = (frame["dow"].isin([5, 6])).astype(int)
    frame["minute_of_day"] = frame["time_bin"].dt.hour * 60 + frame["time_bin"].dt.minute
    sin_h, cos_h = _sin_cos(frame["hour"], 24)
    sin_m, cos_m = _sin_cos(frame["minute_of_day"], 24 * 60)
    frame["sin_hour"], frame["cos_hour"] = sin_h, cos_h
    frame["sin_min"], frame["cos_min"] = sin_m, cos_m
    return frame

def add_lagged_counts(
    frame: pd.DataFrame,
    group_col: str,
    value_col: str = "pickup_cnt",
    lags: Tuple[int, ...] = (1, 2, 3),
    mas: Tuple[int, ...] = (6, 12),
    prefix: str = "pickup",
) -> pd.DataFrame:
    frame = frame.copy()
    frame = frame.sort_values([group_col, "time_bin"])
    def _per_group(g: pd.DataFrame) -> pd.DataFrame:
        g = g.copy()
        for k in lags:
            g[f"{prefix}_lag_{k}"] = g[value_col].shift(k)
        for w in mas:
            g[f"{prefix}_ma_{w}"] = g[value_col].rolling(window=w, min_periods=1).mean()
        return g
    frame = frame.groupby(group_col, group_keys=False).apply(_per_group)
    return frame

def add_neighbor_aggregates(
    frame: pd.DataFrame,
    cell_col: str = "h3_cell",
    value_cols: Tuple[str, ...] = ("pickup_cnt", "pickup_lag_1", "pickup_lag_2", "pickup_lag_3"),
    k_rings: Tuple[int, ...] = (1, 2),
    max_batch_cells: int = 2000,
) -> pd.DataFrame:
    """
    Memory-friendly neighbor aggregates computed in batches.
    Produces columns: nbr{K}_mean_{value_col}
    """
    frame = frame.copy()
    # minimal tmp that we need
    tmp = frame[["time_bin", cell_col] + list(value_cols)].copy()
    unique_cells = [c for c in tmp[cell_col].dropna().unique().tolist()]
    if len(unique_cells) == 0:
        # add zero neighbor columns and return
        for k in k_rings:
            for v in value_cols:
                frame[f"nbr{k}_mean_{v}"] = 0.0
        return frame

    # initialize columns
    for k in k_rings:
        for v in value_cols:
            frame[f"nbr{k}_mean_{v}"] = np.nan

    # index tmp for fast merge lookups
    lookup = tmp.rename(columns={cell_col: "cell_nbr"}).set_index(["time_bin", "cell_nbr"])

    # function to process batch
    def process_batch(batch_cells: List[str]):
        # build map DataFrame: cell_src x neighbor cell_nbr (for max ring to save repeated calls)
        rows = []
        for c in batch_cells:
            rows.append({"cell_src": c, "neighbors": _h3_neighbors(c, max(k_rings))})
        map_df = pd.DataFrame(rows).explode("neighbors").rename(columns={"neighbors": "cell_nbr"})
        # join times by merging with available lookup keys:
        # create a merge-ready DataFrame by repeating map_df for each time_bin (cartesian) would be heavy,
        # so instead iterate over unique time bins — still expensive but batched by cells reduces peak mem.
        time_bins = tmp["time_bin"].drop_duplicates().tolist()
        # For memory reasons keep iteration over time bins (time bins often smaller than cells)
        # build a small list of results to merge back
        results = []
        for tb in time_bins:
            # values available for this time bin
            try:
                vals = lookup.loc[tb]
            except KeyError:
                # no data for this time bin
                continue
            vals = vals.reset_index()  # columns: cell_nbr + value_cols
            # map_df join with vals on cell_nbr
            joined = map_df.merge(vals, on="cell_nbr", how="left")
            if joined.empty:
                continue
            # for each k compute neighbor means by selecting neighbors that fall into k-ring sets
            for k in k_rings:
                # precompute neighbors_k for each cell_src
                nk_map = {c: set(_h3_neighbors(c, k)) for c in batch_cells}
                # filter joined rows by neighbor membership
                # joined has columns: cell_src, cell_nbr, value_cols...
                joined["is_in_k"] = joined.apply(lambda r, nm=nk_map: (r["cell_nbr"] in nm.get(r["cell_src"], set())), axis=1)
                subset = joined[joined["is_in_k"]]
                if subset.empty:
                    # still need to create rows (no neighbors found) -> default NaN
                    continue
                grp = subset.groupby("cell_src")[list(value_cols)].mean().reset_index()
                for v in value_cols:
                    colname = f"nbr{k}_mean_{v}"
                    grp2 = grp[["cell_src", v]].rename(columns={"cell_src": cell_col, v: colname})
                    # attach time_bin for merging back
                    grp2["time_bin"] = tb
                    results.append(grp2[["time_bin", cell_col, colname]])
        if len(results) == 0:
            return
        merged = pd.concat(results, axis=0, ignore_index=True)
        # merge into frame
        frame_cols_to_merge = [c for c in merged.columns if c not in ["time_bin", cell_col]]
        frame_local = frame.merge(merged, on=["time_bin", cell_col], how="left")
        # update frame in place for new columns
        for c in frame_local.columns:
            if c.startswith("nbr") and c in frame.columns:
                frame[c] = frame_local[c]
        return

    # process unique_cells in batches
    n = len(unique_cells)
    for i in range(0, n, max_batch_cells):
        batch = unique_cells[i : i + max_batch_cells]
        process_batch(batch)

    # finally fill any NaNs with 0
    nbr_cols = [c for c in frame.columns if c.startswith("nbr")]
    frame[nbr_cols] = frame[nbr_cols].fillna(0.0)
    return frame

def build_features_safe(
    trips: pd.DataFrame,
    pickup_time_col: str,
    pickup_lat_col: str,
    pickup_lon_col: str,
    dropoff_time_col: Optional[str] = None,
    dropoff_lat_col: Optional[str] = None,
    dropoff_lon_col: Optional[str] = None,
    # h3_res: int = 7,
    h3_res: int = 7,
    start_time: Optional[pd.Timestamp] = None,
    end_time: Optional[pd.Timestamp] = None,
    max_cells_for_full_index: int = 20000,  # safety threshold
    neighbor_batch_cells: int = 2000,
) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    """
    Safer version of build_features:
     - default h3_res=7 for fewer cells
     - guards for empty inputs
     - will raise a clear error if resource limits would be exceeded
    """
    df = trips.copy()
    if df.shape[0] == 0:
        raise ValueError("trips DataFrame is empty")

    df[pickup_time_col] = pd.to_datetime(df[pickup_time_col])
    if start_time is not None:
        df = df[df[pickup_time_col] >= pd.to_datetime(start_time)]
    if end_time is not None:
        df = df[df[pickup_time_col] < pd.to_datetime(end_time)]
    if df.shape[0] == 0:
        raise ValueError("No rows after time filtering")

    # Assign pickup cell (drop rows where assignment failed)
    df = assign_h3_cells(df, pickup_lat_col, pickup_lon_col, h3_res=h3_res, out_col="pickup_cell")
    df = df.dropna(subset=[pickup_time_col, "pickup_cell"])
    if df.shape[0] == 0:
        raise ValueError("No valid pickup events after assigning h3 cell and dropping NaNs")

    # initial aggregate to know extent
    agg0 = aggregate_events_to_cells(df=df, ts_col=pickup_time_col, cell_col="pickup_cell", bin_minutes=5)
    if agg0.empty:
        raise ValueError("No aggregated pickups found (agg0 empty)")

    time_start = agg0["time_bin"].min()
    time_end = agg0["time_bin"].max()
    time_bins = pd.date_range(start=time_start, end=time_end, freq="5min")
    cells = agg0["pickup_cell"].dropna().unique().tolist()
    if len(cells) == 0:
        raise ValueError("No h3 cells found in pickups")

    # safety guard: if cells * time_bins huge, abort with an informative error
    est_rows = len(cells) * len(time_bins)
    if est_rows > 50_000_000:  # arbitrary big threshold; adjust if you want risk
        raise MemoryError(f"Estimated full grid size {est_rows:,} > 50M rows. Reduce h3_res or window.")

    if len(cells) > max_cells_for_full_index:
        raise MemoryError(f"Number of cells ({len(cells):,}) > max_cells_for_full_index ({max_cells_for_full_index}). "
                          "Consider lowering h3_res, restricting bounding box, or sampling.")

    full_idx = build_complete_index(time_bins, cells)

    pickup_grid = aggregate_events_to_cells(
        df=df[[pickup_time_col, "pickup_cell"]].rename(columns={pickup_time_col: "ts", "pickup_cell": "h3_cell"}).dropna(),
        ts_col="ts",
        cell_col="h3_cell",
        bin_minutes=5,
        complete_index=full_idx,
    ).rename(columns={"value": "pickup_cnt"})

    # dropoffs (supply) — optional
    if dropoff_lat_col and dropoff_lon_col and dropoff_time_col and all(col in df.columns for col in [dropoff_lat_col, dropoff_lon_col, dropoff_time_col]):
        ddf = df[[dropoff_time_col, dropoff_lat_col, dropoff_lon_col]].dropna().copy()
        ddf = assign_h3_cells(ddf, dropoff_lat_col, dropoff_lon_col, h3_res=h3_res, out_col="drop_cell")
        drop_grid = aggregate_events_to_cells(
            df=ddf,
            ts_col=dropoff_time_col,
            cell_col="drop_cell",
            bin_minutes=5,
            complete_index=full_idx,
        ).rename(columns={"drop_cell": "h3_cell", "value": "dropoff_cnt"})
    else:
        drop_grid = None

    base = pickup_grid.copy()
    if drop_grid is not None:
        base = base.merge(drop_grid, on=["time_bin", "h3_cell"], how="left")
        base["dropoff_cnt"] = base["dropoff_cnt"].fillna(0)
    else:
        base["dropoff_cnt"] = 0

    base = add_time_features(base)
    base = add_lagged_counts(base, group_col="h3_cell", value_col="pickup_cnt", lags=(1,2,3), mas=(6,12), prefix="pickup")
    base = add_lagged_counts(base, group_col="h3_cell", value_col="dropoff_cnt", lags=(1,2,3), mas=(6,12), prefix="dropoff")
    base["net_inflow"] = base["dropoff_cnt"] - base["pickup_cnt"]

    # neighbor aggregates (batched)
    base = add_neighbor_aggregates(
        base,
        cell_col="h3_cell",
        value_cols=("pickup_cnt", "pickup_lag_1", "pickup_lag_2", "pickup_lag_3", "dropoff_cnt"),
        k_rings=(1,2),
        max_batch_cells=neighbor_batch_cells,
    )

    base = base.sort_values(["h3_cell", "time_bin"]).reset_index(drop=True)
    base["future_pickup_cnt"] = base.groupby("h3_cell")["pickup_cnt"].shift(-1)
    target = (base["future_pickup_cnt"].fillna(0) > 0).astype(int).rename("y_next5_has_pickup")

    feature_cols = [
        "hour", "dow", "is_weekend", "sin_hour", "cos_hour", "sin_min", "cos_min",
        "pickup_cnt", "dropoff_cnt", "net_inflow",
        "pickup_lag_1", "pickup_lag_2", "pickup_lag_3", "pickup_ma_6", "pickup_ma_12",
        "dropoff_lag_1", "dropoff_lag_2", "dropoff_lag_3", "dropoff_ma_6", "dropoff_ma_12",
        "nbr1_mean_pickup_cnt", "nbr2_mean_pickup_cnt",
        "nbr1_mean_pickup_lag_1", "nbr2_mean_pickup_lag_1",
        "nbr1_mean_pickup_lag_2", "nbr2_mean_pickup_lag_2",
        "nbr1_mean_pickup_lag_3", "nbr2_mean_pickup_lag_3",
        "nbr1_mean_dropoff_cnt", "nbr2_mean_dropoff_cnt",
    ]
    for c in feature_cols:
        if c not in base.columns:
            base[c] = 0.0

    X = base[feature_cols].fillna(0.0)
    meta = base[["time_bin", "h3_cell", "pickup_cnt"]].copy()
    return X, target, meta

# quick helper alias to preserve old name if you want:
build_features = build_features_safe

print("Safe H3 5-minute feature utilities loaded (build_features). Recommended defaults: h3_res=7 on 16GB MacBook Air.")


Safe H3 5-minute feature utilities loaded (build_features). Recommended defaults: h3_res=7 on 16GB MacBook Air.


In [ ]:
# MEMORY-OPTIMIZED PROCESSING - PREVENTS CRASHES WHILE KEEPING h3_res=7 and k_rings=(1,2)
# Enhanced version with better memory management and smaller processing chunks

import gc
import pandas as pd
from pathlib import Path
import numpy as np
import psutil
import os

def get_memory_usage():
    """Get current memory usage in MB"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

def get_busiest_cells_per_window(trips_df, h3_res=7, top_n=200):
    """Get top N busiest cells for this time window - reduced from 200 to 150"""
    # Assign cells
    trips_with_cells = assign_h3_cells(
        trips_df.copy(), 
        "pickup_latitude", 
        "pickup_longitude", 
        h3_res=h3_res, 
        out_col="pickup_cell"
    )
    # Count pickups per cell
    cell_counts = trips_with_cells["pickup_cell"].value_counts()
    # Return top N cells
    top_cells = cell_counts.head(top_n).index.tolist()
    return top_cells

def add_neighbor_aggregates_optimized(
    frame: pd.DataFrame,
    cell_col: str = "h3_cell",
    value_cols: tuple = ("pickup_cnt", "pickup_lag_1", "pickup_lag_2", "pickup_lag_3"),
    k_rings: tuple = (1, 2),
    max_batch_cells: int = 60,  # Much smaller batches
    memory_threshold_mb: int = 10000,  # Stop if memory gets too high
) -> pd.DataFrame:
    """
    Memory-optimized version of add_neighbor_aggregates
    """
    frame = frame.copy()
    
    # Check initial memory
    initial_memory = get_memory_usage()
    print(f"  Initial memory: {initial_memory:.1f} MB")
    
    # minimal tmp that we need
    tmp = frame[["time_bin", cell_col] + list(value_cols)].copy()
    unique_cells = [c for c in tmp[cell_col].dropna().unique().tolist()]
    
    if len(unique_cells) == 0:
        # add zero neighbor columns and return
        for k in k_rings:
            for v in value_cols:
                frame[f"nbr{k}_mean_{v}"] = 0.0
        return frame

    # initialize columns
    for k in k_rings:
        for v in value_cols:
            frame[f"nbr{k}_mean_{v}"] = np.nan

    # index tmp for fast merge lookups
    lookup = tmp.rename(columns={cell_col: "cell_nbr"}).set_index(["time_bin", "cell_nbr"])

    # function to process batch with memory monitoring
    def process_batch_optimized(batch_cells: list):
        current_memory = get_memory_usage()
        if current_memory > memory_threshold_mb:
            print(f"  ⚠️ Memory threshold reached ({current_memory:.1f} MB), skipping batch")
            return
            
        # build map DataFrame: cell_src x neighbor cell_nbr (for max ring to save repeated calls)
        rows = []
        for c in batch_cells:
            try:
                neighbors = _h3_neighbors(c, max(k_rings))
                rows.append({"cell_src": c, "neighbors": neighbors})
            except Exception as e:
                print(f"  ⚠️ Error getting neighbors for cell {c}: {e}")
                continue
                
        if not rows:
            return
            
        map_df = pd.DataFrame(rows).explode("neighbors").rename(columns={"neighbors": "cell_nbr"})
        
        # Process time bins in smaller chunks
        time_bins = tmp["time_bin"].drop_duplicates().tolist()
        results = []
        
        # Process time bins in smaller chunks to reduce memory
        time_chunk_size = min(100, len(time_bins))  # Process max 50 time bins at once
        
        for i in range(0, len(time_bins), time_chunk_size):
            time_chunk = time_bins[i:i + time_chunk_size]
            
            for tb in time_chunk:
                # values available for this time bin
                try:
                    vals = lookup.loc[tb]
                except KeyError:
                    continue
                    
                vals = vals.reset_index()
                
                # map_df join with vals on cell_nbr
                joined = map_df.merge(vals, on="cell_nbr", how="left")
                if joined.empty:
                    continue
                    
                # for each k compute neighbor means by selecting neighbors that fall into k-ring sets
                for k in k_rings:
                    # precompute neighbors_k for each cell_src
                    nk_map = {}
                    for c in batch_cells:
                        try:
                            nk_map[c] = set(_h3_neighbors(c, k))
                        except Exception as e:
                            print(f"  ⚠️ Error getting k={k} neighbors for cell {c}: {e}")
                            continue
                    
                    # filter joined rows by neighbor membership using vectorized operations
                    if nk_map:
                        # Create a more efficient membership check
                        joined["is_in_k"] = False
                        for cell_src, neighbors in nk_map.items():
                            mask = joined["cell_src"] == cell_src
                            joined.loc[mask, "is_in_k"] = joined.loc[mask, "cell_nbr"].isin(neighbors)
                        
                        subset = joined[joined["is_in_k"]]
                        if subset.empty:
                            continue
                            
                        grp = subset.groupby("cell_src")[list(value_cols)].mean().reset_index()
                        for v in value_cols:
                            colname = f"nbr{k}_mean_{v}"
                            grp2 = grp[["cell_src", v]].rename(columns={"cell_src": cell_col, v: colname})
                            grp2["time_bin"] = tb
                            results.append(grp2[["time_bin", cell_col, colname]])
            
            # Force garbage collection after each time chunk
            gc.collect()
            
        if len(results) == 0:
            return
            
        try:
            merged = pd.concat(results, axis=0, ignore_index=True)
            # merge into frame
            frame_cols_to_merge = [c for c in merged.columns if c not in ["time_bin", cell_col]]
            frame_local = frame.merge(merged, on=["time_bin", cell_col], how="left")
            # update frame in place for new columns
            for c in frame_local.columns:
                if c.startswith("nbr") and c in frame.columns:
                    frame[c] = frame_local[c]
        except Exception as e:
            print(f"  ⚠️ Error merging neighbor results: {e}")

    # process unique_cells in very small batches
    n = len(unique_cells)
    batch_size = min(max_batch_cells, 60)  # Even smaller batches
    
    for i in range(0, n, batch_size):
        batch = unique_cells[i : i + batch_size]
        print(f"  Processing cell batch {i//batch_size + 1}/{(n-1)//batch_size + 1} ({len(batch)} cells)")
        
        process_batch_optimized(batch)
        
        # Force garbage collection after each batch
        gc.collect()
        
        # Check memory after each batch
        current_memory = get_memory_usage()
        if current_memory > memory_threshold_mb:
            print(f"  ⚠️ Memory limit reached, stopping neighbor processing")
            break

    # finally fill any NaNs with 0
    nbr_cols = [c for c in frame.columns if c.startswith("nbr")]
    frame[nbr_cols] = frame[nbr_cols].fillna(0.0)
    return frame

def build_features_memory_optimized(
    trips: pd.DataFrame,
    pickup_time_col: str,
    pickup_lat_col: str,
    pickup_lon_col: str,
    dropoff_time_col: str = None,
    dropoff_lat_col: str = None,
    dropoff_lon_col: str = None,
    h3_res: int = 7,
    start_time: pd.Timestamp = None,
    end_time: pd.Timestamp = None,
    max_cells_for_full_index: int = 180,  # Reduced from 200
    neighbor_batch_cells: int = 30,   # Reduced from 50
) -> tuple:
    """
    Memory-optimized version of build_features that prevents crashes
    """
    print(f"  Building features with h3_res={h3_res}, max_cells={max_cells_for_full_index}")
    print(f"  Initial memory: {get_memory_usage():.1f} MB")
    
    df = trips.copy()
    if df.shape[0] == 0:
        raise ValueError("trips DataFrame is empty")

    df[pickup_time_col] = pd.to_datetime(df[pickup_time_col])
    if start_time is not None:
        df = df[df[pickup_time_col] >= pd.to_datetime(start_time)]
    if end_time is not None:
        df = df[df[pickup_time_col] < pd.to_datetime(end_time)]
    if df.shape[0] == 0:
        raise ValueError("No rows after time filtering")

    # Assign pickup cell (drop rows where assignment failed)
    df = assign_h3_cells(df, pickup_lat_col, pickup_lon_col, h3_res=h3_res, out_col="pickup_cell")
    df = df.dropna(subset=[pickup_time_col, "pickup_cell"])
    if df.shape[0] == 0:
        raise ValueError("No valid pickup events after assigning h3 cell and dropping NaNs")

    # initial aggregate to know extent
    agg0 = aggregate_events_to_cells(df=df, ts_col=pickup_time_col, cell_col="pickup_cell", bin_minutes=5)
    if agg0.empty:
        raise ValueError("No aggregated pickups found (agg0 empty)")

    time_start = agg0["time_bin"].min()
    time_end = agg0["time_bin"].max()
    time_bins = pd.date_range(start=time_start, end=time_end, freq="5min")
    cells = agg0["pickup_cell"].dropna().unique().tolist()
    if len(cells) == 0:
        raise ValueError("No h3 cells found in pickups")

    # safety guard: if cells * time_bins huge, abort with an informative error
    est_rows = len(cells) * len(time_bins)
    if est_rows > 10_000_000:  # Reduced threshold
        raise MemoryError(f"Estimated full grid size {est_rows:,} > 10M rows. Reduce h3_res or window.")

    if len(cells) > max_cells_for_full_index:
        raise MemoryError(f"Number of cells ({len(cells):,}) > max_cells_for_full_index ({max_cells_for_full_index}). "
                          "Consider lowering h3_res, restricting bounding box, or sampling.")

    full_idx = build_complete_index(time_bins, cells)

    pickup_grid = aggregate_events_to_cells(
        df=df[[pickup_time_col, "pickup_cell"]].rename(columns={pickup_time_col: "ts", "pickup_cell": "h3_cell"}).dropna(),
        ts_col="ts",
        cell_col="h3_cell",
        bin_minutes=5,
        complete_index=full_idx,
    ).rename(columns={"value": "pickup_cnt"})

    # dropoffs (supply) — optional
    if dropoff_lat_col and dropoff_lon_col and dropoff_time_col and all(col in df.columns for col in [dropoff_lat_col, dropoff_lon_col, dropoff_time_col]):
        ddf = df[[dropoff_time_col, dropoff_lat_col, dropoff_lon_col]].dropna().copy()
        ddf = assign_h3_cells(ddf, dropoff_lat_col, dropoff_lon_col, h3_res=h3_res, out_col="drop_cell")
        drop_grid = aggregate_events_to_cells(
            df=ddf,
            ts_col=dropoff_time_col,
            cell_col="drop_cell",
            bin_minutes=5,
            complete_index=full_idx,
        ).rename(columns={"drop_cell": "h3_cell", "value": "dropoff_cnt"})
    else:
        drop_grid = None

    base = pickup_grid.copy()
    if drop_grid is not None:
        base = base.merge(drop_grid, on=["time_bin", "h3_cell"], how="left")
        base["dropoff_cnt"] = base["dropoff_cnt"].fillna(0)
    else:
        base["dropoff_cnt"] = 0

    base = add_time_features(base)
    base = add_lagged_counts(base, group_col="h3_cell", value_col="pickup_cnt", lags=(1,2,3), mas=(6,12), prefix="pickup")
    base = add_lagged_counts(base, group_col="h3_cell", value_col="dropoff_cnt", lags=(1,2,3), mas=(6,12), prefix="dropoff")
    base["net_inflow"] = base["dropoff_cnt"] - base["pickup_cnt"]

    print(f"  Memory before neighbor processing: {get_memory_usage():.1f} MB")
    
    # Use optimized neighbor aggregates
    base = add_neighbor_aggregates_optimized(
        base,
        cell_col="h3_cell",
        value_cols=("pickup_cnt", "pickup_lag_1", "pickup_lag_2", "pickup_lag_3", "dropoff_cnt"),
        k_rings=(1, 2),  # Keep k_rings=(1,2) as requested
        max_batch_cells=neighbor_batch_cells,
    )

    base = base.sort_values(["h3_cell", "time_bin"]).reset_index(drop=True)
    base["future_pickup_cnt"] = base.groupby("h3_cell")["pickup_cnt"].shift(-1)
    target = (base["future_pickup_cnt"].fillna(0) > 0).astype(int).rename("y_next5_has_pickup")

    feature_cols = [
        "hour", "dow", "is_weekend", "sin_hour", "cos_hour", "sin_min", "cos_min",
        "pickup_cnt", "dropoff_cnt", "net_inflow",
        "pickup_lag_1", "pickup_lag_2", "pickup_lag_3", "pickup_ma_6", "pickup_ma_12",
        "dropoff_lag_1", "dropoff_lag_2", "dropoff_lag_3", "dropoff_ma_6", "dropoff_ma_12",
        "nbr1_mean_pickup_cnt", "nbr2_mean_pickup_cnt",
        "nbr1_mean_pickup_lag_1", "nbr2_mean_pickup_lag_1",
        "nbr1_mean_pickup_lag_2", "nbr2_mean_pickup_lag_2",
        "nbr1_mean_pickup_lag_3", "nbr2_mean_pickup_lag_3",
        "nbr1_mean_dropoff_cnt", "nbr2_mean_dropoff_cnt",
    ]
    for c in feature_cols:
        if c not in base.columns:
            base[c] = 0.0

    X = base[feature_cols].fillna(0.0)
    meta = base[["time_bin", "h3_cell", "pickup_cnt"]].copy()
    
    print(f"  Final memory: {get_memory_usage():.1f} MB")
    return X, target, meta

def process_time_window_optimized(trips_window, h3_res=7, max_cells=150):
    """Process a single time window with optimized memory management"""
    if trips_window.empty:
        return None, None, None
    
    # Get busiest cells for this window only
    top_cells = get_busiest_cells_per_window(trips_window, h3_res=h3_res, top_n=max_cells)
    
    if not top_cells:
        return None, None, None
    
    # Filter trips to only include top cells
    trips_with_cells = assign_h3_cells(
        trips_window.copy(), 
        "pickup_latitude", 
        "pickup_longitude", 
        h3_res=h3_res, 
        out_col="pickup_cell"
    )
    trips_filtered = trips_with_cells[trips_with_cells["pickup_cell"].isin(top_cells)].copy()
    
    if trips_filtered.empty:
        return None, None, None
    
    print(f"  Processing {len(trips_filtered)} trips in {len(top_cells)} cells")
    
    try:
        X, y, meta = build_features_memory_optimized(
            trips=trips_filtered,
            pickup_time_col="pickup_datetime",
            pickup_lat_col="pickup_latitude",
            pickup_lon_col="pickup_longitude", 
            dropoff_time_col=None,
            dropoff_lat_col=None,
            dropoff_lon_col=None,
            h3_res=h3_res,
            max_cells_for_full_index=max_cells,
            neighbor_batch_cells=30,  # Small batches
        )
        return X, y, meta
    except Exception as e:
        print(f"  ❌ Window failed: {e}")
        return None, None, None
    
# Ensure 'trips' exists with the expected columns
if 'trips' not in globals():
    if 'df' in globals():
        trips = df.rename(columns={
            'timestamp': 'pickup_datetime',
            'lat': 'pickup_latitude',
            'lon': 'pickup_longitude'
        })[['pickup_datetime', 'pickup_latitude', 'pickup_longitude']].dropna()
    elif 'taxis_df' in globals():
        trips = taxis_df.rename(columns={
            'timestamp': 'pickup_datetime',
            'lat': 'pickup_latitude',
            'lon': 'pickup_longitude'
        })[['pickup_datetime', 'pickup_latitude', 'pickup_longitude']].dropna()
    else:
        raise NameError("Define 'trips' with columns: pickup_datetime, pickup_latitude, pickup_longitude.")

trips['pickup_datetime'] = pd.to_datetime(trips['pickup_datetime'])

print("=== MEMORY-OPTIMIZED PROCESSING ===")
print(f"Total trips to process: {len(trips)}")
print(f"Initial memory: {get_memory_usage():.1f} MB")

# Create 2-hour time windows
trips["pickup_datetime"] = pd.to_datetime(trips["pickup_datetime"])
start_time = trips["pickup_datetime"].min()
end_time = trips["pickup_datetime"].max()

# Create 2-hour windows
window_hours = 2
windows = pd.date_range(start_time, end_time, freq=f"{window_hours}H")

print(f"Created {len(windows)-1} time windows of {window_hours} hours each")

# Output files
output_dir = Path("/Users/jul/Desktop/uni/Data Analytics/Taxi-Income-Optimizer/PickUP rate/prepared_data")
output_dir.mkdir(exist_ok=True)

X_path = output_dir / "X_features_5min_optimized.csv"
y_path = output_dir / "y_target_5min_optimized.csv" 
meta_path = output_dir / "meta_5min_optimized.csv"

# Process each window
all_X = []
all_y = []
all_meta = []

for i in range(len(windows)-1):
    window_start = windows[i]
    window_end = windows[i+1]
    
    print(f"\n--- Window {i+1}/{len(windows)-1}: {window_start} to {window_end} ---")
    print(f"Memory before window: {get_memory_usage():.1f} MB")
    
    # Filter trips for this window
    window_trips = trips[
        (trips["pickup_datetime"] >= window_start) & 
        (trips["pickup_datetime"] < window_end)
    ].copy()
    
    if window_trips.empty:
        print("  No trips in this window, skipping")
        continue
    
    print(f"  Found {len(window_trips)} trips in window")
    
    # Process window with optimized function
    X, y, meta = process_time_window_optimized(window_trips, h3_res=7, max_cells=150)
    
    if X is not None:
        all_X.append(X)
        all_y.append(y)
        all_meta.append(meta)
        print(f"  ✅ Generated {len(X)} feature rows")
    else:
        print(f"  ❌ Window failed")
    
    # Force garbage collection
    del window_trips
    gc.collect()
    
    print(f"Memory after window: {get_memory_usage():.1f} MB")

print(f"\n=== RESULTS ===")
if all_X:
    print(f"Successfully processed {len(all_X)} windows")
    print(f"Total feature rows: {sum(len(x) for x in all_X)}")
    
    # Combine all results
    print("Combining results...")
    X_combined = pd.concat(all_X, ignore_index=True)
    y_combined = pd.concat(all_y, ignore_index=True)
    meta_combined = pd.concat(all_meta, ignore_index=True)
    
    # Save to files
    print(f"Saving to files...")
    X_combined.to_csv(X_path, index=False)
    y_combined.to_csv(y_path, index=False)
    meta_combined.to_csv(meta_path, index=False)
    
    print(f"✅ Saved {len(X_combined)} feature rows to:")
    print(f"  - {X_path}")
    print(f"  - {y_path}")
    print(f"  - {meta_path}")
else:
    print("❌ No windows processed successfully")


=== MEMORY-OPTIMIZED PROCESSING ===
Total trips to process: 11651983
Initial memory: 1169.3 MB
Created 640 time windows of 2 hours each

--- Window 1/640: 2024-11-08 15:01:45 to 2024-11-08 17:01:45 ---
Memory before window: 1272.8 MB
  Found 3 trips in window
  Processing 3 trips in 1 cells
  Building features with h3_res=7, max_cells=150
  Initial memory: 1273.1 MB
  Memory before neighbor processing: 1273.4 MB
  Initial memory: 1273.4 MB
  Processing cell batch 1/1 (1 cells)


/var/folders/qb/yf4211gs4kv7hy5d5jp7q6bw0000gn/T/ipykernel_2481/2071981944.py:391: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  windows = pd.date_range(start_time, end_time, freq=f"{window_hours}H")
/var/folders/qb/yf4211gs4kv7hy5d5jp7q6bw0000gn/T/ipykernel_2481/1425735835.py:111: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  frame = frame.groupby(group_col, group_keys=False).apply(_per_group)
/var/folders/qb/yf4211gs4kv7hy5d5jp7q6bw0000gn/T/ipykernel_2481/1425735835.py:111: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be e

  Final memory: 1274.1 MB
  ✅ Generated 1 feature rows
Memory after window: 1274.2 MB

--- Window 2/640: 2024-11-08 17:01:45 to 2024-11-08 19:01:45 ---
Memory before window: 1274.2 MB
  No trips in this window, skipping

--- Window 3/640: 2024-11-08 19:01:45 to 2024-11-08 21:01:45 ---
Memory before window: 1274.2 MB
  No trips in this window, skipping

--- Window 4/640: 2024-11-08 21:01:45 to 2024-11-08 23:01:45 ---
Memory before window: 1274.2 MB
  No trips in this window, skipping

--- Window 5/640: 2024-11-08 23:01:45 to 2024-11-09 01:01:45 ---
Memory before window: 1274.2 MB
  No trips in this window, skipping

--- Window 6/640: 2024-11-09 01:01:45 to 2024-11-09 03:01:45 ---
Memory before window: 1274.2 MB
  No trips in this window, skipping

--- Window 7/640: 2024-11-09 03:01:45 to 2024-11-09 05:01:45 ---
Memory before window: 1274.2 MB
  No trips in this window, skipping

--- Window 8/640: 2024-11-09 05:01:45 to 2024-11-09 07:01:45 ---
Memory before window: 1274.2 MB
  No trips i

/var/folders/qb/yf4211gs4kv7hy5d5jp7q6bw0000gn/T/ipykernel_2481/1425735835.py:111: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  frame = frame.groupby(group_col, group_keys=False).apply(_per_group)
/var/folders/qb/yf4211gs4kv7hy5d5jp7q6bw0000gn/T/ipykernel_2481/1425735835.py:111: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  frame = frame.groupby(group_col, group_keys=False).apply(_per_group)


  No trips in this window, skipping

--- Window 61/640: 2024-11-13 15:01:45 to 2024-11-13 17:01:45 ---
Memory before window: 1274.5 MB
  No trips in this window, skipping

--- Window 62/640: 2024-11-13 17:01:45 to 2024-11-13 19:01:45 ---
Memory before window: 1274.5 MB
  No trips in this window, skipping

--- Window 63/640: 2024-11-13 19:01:45 to 2024-11-13 21:01:45 ---
Memory before window: 1274.5 MB
  No trips in this window, skipping

--- Window 64/640: 2024-11-13 21:01:45 to 2024-11-13 23:01:45 ---
Memory before window: 1274.5 MB
  No trips in this window, skipping

--- Window 65/640: 2024-11-13 23:01:45 to 2024-11-14 01:01:45 ---
Memory before window: 1274.5 MB
  No trips in this window, skipping

--- Window 66/640: 2024-11-14 01:01:45 to 2024-11-14 03:01:45 ---
Memory before window: 1274.5 MB
  No trips in this window, skipping

--- Window 67/640: 2024-11-14 03:01:45 to 2024-11-14 05:01:45 ---
Memory before window: 1274.5 MB
  No trips in this window, skipping

--- Window 68/640:

...

/var/folders/qb/yf4211gs4kv7hy5d5jp7q6bw0000gn/T/ipykernel_2481/1425735835.py:111: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  frame = frame.groupby(group_col, group_keys=False).apply(_per_group)
/var/folders/qb/yf4211gs4kv7hy5d5jp7q6bw0000gn/T/ipykernel_2481/1425735835.py:111: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  frame = frame.groupby(group_col, group_keys=False).apply(_per_group)


  Memory before neighbor processing: 437.0 MB
  Initial memory: 437.0 MB
  Processing cell batch 1/5 (30 cells)
  Processing cell batch 2/5 (30 cells)
  Processing cell batch 3/5 (30 cells)
  Processing cell batch 4/5 (30 cells)
  Processing cell batch 5/5 (30 cells)
  Final memory: 434.5 MB
  ✅ Generated 3750 feature rows
Memory after window: 434.5 MB

--- Window 640/640: 2024-12-31 21:01:45 to 2024-12-31 23:01:45 ---
Memory before window: 434.5 MB
  Found 32581 trips in window
  Processing 27289 trips in 150 cells
  Building features with h3_res=7, max_cells=150
  Initial memory: 435.7 MB


/var/folders/qb/yf4211gs4kv7hy5d5jp7q6bw0000gn/T/ipykernel_2481/1425735835.py:111: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  frame = frame.groupby(group_col, group_keys=False).apply(_per_group)
/var/folders/qb/yf4211gs4kv7hy5d5jp7q6bw0000gn/T/ipykernel_2481/1425735835.py:111: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  frame = frame.groupby(group_col, group_keys=False).apply(_per_group)


  Memory before neighbor processing: 435.7 MB
  Initial memory: 435.7 MB
  Processing cell batch 1/5 (30 cells)
  Processing cell batch 2/5 (30 cells)
  Processing cell batch 3/5 (30 cells)
  Processing cell batch 4/5 (30 cells)
  Processing cell batch 5/5 (30 cells)
  Final memory: 436.0 MB
  ✅ Generated 3750 feature rows
Memory after window: 436.0 MB

=== RESULTS ===
Successfully processed 382 windows
Total feature rows: 1394272
Combining results...
Saving to files...
✅ Saved 1394272 feature rows to:
  - /Users/jul/Desktop/uni/Data Analytics/Taxi-Income-Optimizer/PickUP rate/prepared_data/X_features_5min_optimized.csv
  - /Users/jul/Desktop/uni/Data Analytics/Taxi-Income-Optimizer/PickUP rate/prepared_data/y_target_5min_optimized.csv
  - /Users/jul/Desktop/uni/Data Analytics/Taxi-Income-Optimizer/PickUP rate/prepared_data/meta_5min_optimized.csv


: 